In [4]:
import importlib
import src.backtesting as bt

# Reload the latest saved version
importlib.reload(bt)

# Display available functions
print("Module:", bt.__file__)
print("run_backtest exists:", hasattr(bt, "run_backtest"))
print("Available names:", [
    name for name in dir(bt)
    if not name.startswith("_")
])

Module: C:\Users\kyan\OneDrive\Documents\Projects\ForecastGuard\src\backtesting.py
run_backtest exists: True
Available names: ['DEFAULT_CUTOFFS', 'FEATURES', 'build_quantile_model', 'inventory_cost', 'np', 'optimal_quantile', 'pd', 'run_backtest']


In [5]:

from pathlib import Path
import sys
import pandas as pd

# Find the project root
project_root = Path.cwd().resolve()

while not (
    project_root / "data/selected_product_sales.csv"
).exists():

    parent = project_root.parent

    if parent == project_root:
        raise FileNotFoundError("ForecastGuard folder not found")

    project_root = parent

sys.path.insert(0, str(project_root))

# Import reusable modules
from src.data_pipeline import prepare_product_data
from src.backtesting import run_backtest

# Prepare dataset
df = prepare_product_data(project_root)

# Reproduce the 80th-percentile experiment
backtest, predictions = run_backtest(
    df,
    overstock_cost=2,
    understock_cost=8
)

print("BACKTEST RESULTS")
print(backtest.round(2).to_string(index=False))

# Aggregate performance
ml_total = backtest["ML Cost"].sum()
seasonal_total = backtest["Seasonal Cost"].sum()

savings = (
    (seasonal_total - ml_total)
    / seasonal_total * 100
)

print("\nOVERALL RESULTS")
print(f"ML Cost: ${ml_total:,.2f}")
print(f"Seasonal Cost: ${seasonal_total:,.2f}")
print(f"Simulated Cost Reduction: {savings:.2f}%")

print(
    "Winning Folds:",
    (backtest["Savings (%)"] > 0).sum(),
    "/", len(backtest)
)

print("Total predictions:", len(predictions))


c:\Users\kyan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\kyan\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\kyan\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kyan\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^

BACKTEST RESULTS
 Fold  Start Day  End Day  ML Cost  Seasonal Cost  Savings (%)  Quantile Coverage (%)
    1       1494     1549  12108.0        14292.0        15.28                  78.93
    2       1550     1605  10930.0        16696.0        34.54                  80.71
    3       1606     1661  11104.0        14286.0        22.27                  78.93
    4       1662     1717  10306.0        10304.0        -0.02                  84.29
    5       1718     1773  12948.0        16118.0        19.67                  80.00
    6       1774     1829   9840.0        16472.0        40.26                  83.57
    7       1830     1885  10994.0        13130.0        16.27                  80.36
    8       1886     1941   9746.0        10980.0        11.24                  82.86

OVERALL RESULTS
ML Cost: $87,976.00
Seasonal Cost: $112,278.00
Simulated Cost Reduction: 21.64%
Winning Folds: 7 / 8
Total predictions: 2240


In [6]:

# Create results directory
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

# Save fold-level results
backtest.to_csv(
    results_dir / "backtest_summary.csv",
    index=False
)

# Save individual model predictions
predictions.to_csv(
    results_dir / "backtest_predictions.csv",
    index=False
)

print("Results saved successfully!")


Results saved successfully!


In [7]:

import numpy as np

assert len(predictions) == 2240

assert not predictions.duplicated(
    ["item_id", "day_number"]
).any()

assert np.isclose(
    predictions["ml_cost"].sum(),
    backtest["ML Cost"].sum()
)

assert np.isclose(
    predictions["seasonal_cost"].sum(),
    backtest["Seasonal Cost"].sum()
)

print("Backtest integrity checks passed!")


Backtest integrity checks passed!
